In [ ]:
from abc import ABC, abstractmethod
class Quacker(ABC):  # this tells that Quacker is an abstract class
                     # and tells any class that claims to be Quacker must have a quack method.
    @abstractmethod
    def quack(self):
        pass

class Duck:
    def quack(self):
        return "Quack!"
class Cat:
    def quack(self):
        return "Meow!"
       
def is_a_quacker(obj):
    if type(obj) is Quacker:
        return True
    return False
# - This function checks if an object is exactly of type Quacker. But this won’t work as expected because:
# - Duck and Cat are not subclasses of Quacker.
# - Even if they behave like Quackers, type(obj) only checks direct type, not behavior.

Quacker.register(Duck)  # This is the magic! It tells Python: “Treat Duck as a virtual subclass of Quacker.”

print(isinstance(duck, Quacker))  # True
print(isinstance(cat, Quacker))   # False

In [ ]:
import threading

### The SharedCounter class
class SharedCounter:
    def __init__(self):
        self.value = 0   # this is an integer that all threads will modify
        self._value_lock = threading.Lock()  # a mutex lock that ensures only one thread can modify value at a time

    def increment(self, delta=1):
        with self._value_lock:  # adds delta to value while holding the lock
            self.value += delta

    def get_value(self):
        with self._value_lock:  # reads value while holding the lock
            return self.value
        
### Multithreaded Execution
def worker(counter, num_iters): # function to increment counter num_iters times
    for _ in range(num_iters):
        counter.increment()

if __name__ == "__main__":
    counter = SharedCounter()   # create sharedcounter object
    num_iters = 10000           # each thread will increament counter 10,000 times
    num_workers = 5             # 5 threads
    
    # create list of 5 threads, each running worker(counter, num_iters)
    threads = [
        threading.Thread(target=worker,
                         args=(counter, num_iters)) for _ in range(num_workers)
                         ]
    for t in threads:
        t.start()  # start each thread
    for t in threads:
        t.join()   # wait for each thread to finish
    print(counter.get_value())

In [ ]:
# Deep learning models for NER using Keras
# combining Bi-directional LSTM with a CRF (Conditional Random Field)

from keras.models import Sequential
from keras.layers import Embedding, Bidirectional, LSTM, Dense, TimeDistributed, CRF

model = Sequential()
model.add(Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_seq_length))
model.add(Bidirectional(LSTM(units=lstm_units, return_sequences=True)))
model.add(TimeDistributed(Dense(num_labels)))
model.add(CRF(num_labels))

## Modelling Code

In [ ]:
import pandas as pd

train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

In [ ]:
from keras.preprocessing.text import Tokenizer
# from keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.sequence import pad_sequences

max_vocab_size = 10000
max_seq_length = 100

tokenizer = Tokenizer(num_words=max_vocab_size, 
                      oov_token="<OOV>")
tokenizer.fit_on_texts(train_df['customer_review'])

X_train = tokenizer.texts_to_sequences(train_df['customer_review'])
X_train = pad_sequences(X_train, 
                        maxlen=max_seq_length, 
                        padding='post')

X_test = tokenizer.texts_to_sequences(test_df['customer_review'])
X_test = pad_sequences(X_test, 
                       maxlen=max_seq_length, 
                       padding='post')
y_train = train_df['feedback'].values

In [ ]:
from keras.models import Sequential
from keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional

embedding_dim = 64

model = Sequential([
    Embedding(input_dim=max_vocab_size, 
              output_dim=embedding_dim, 
              input_length=max_seq_length),
    Bidirectional(LSTM(64, return_sequences=False)),
    Dropout(0.5),
    Dense(32, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid')
])

In [ ]:
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

In [ ]:
from keras.callbacks import EarlyStopping

early_stopping = EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)

In [ ]:
from sklearn.model_selection import train_test_split

X_train_sub, X_val, y_train_sub, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

model.fit(
    X_train_sub, y_train_sub,
    epochs=10,
    batch_size=128,
    validation_data=(X_val, y_val),
    callbacks=[early_stopping]
)

In [ ]:
y_pred_probs = model.predict(X_test)
y_pred = (y_pred_probs > 0.5).astype(int).flatten()